# 02 · Construcción de muestra, panel no balanceado y balanceado

**Objetivo:** separar población observada, criterios de elegibilidad y abandono del seguimiento. **No asumir** que panel completo es la muestra principal. Ejecutar primero el 01 o cargar nuevamente el archivo.


In [ ]:
from tesis_enasem.config import RAW_DIR, ROUND_TO_YEAR
from tesis_enasem.io import load_stata
from tesis_enasem.panel import make_person_id, add_year, filter_population, balanced_panel_ids
from tesis_enasem.missing import numeric_enasem
from tesis_enasem.validation import panel_quality
import pandas as pd

raw=load_stata(RAW_DIR/'simpleMHAS.dta', preserve_missing=True)
raw['id']=make_person_id(raw)
raw=add_year(raw, ROUND_TO_YEAR)


## Flujo de entrevistas y observaciones
Los filtros de entrevista directa (1,2) eliminan casos sustitutos/fallecidos de la muestra de predictores pero **no** de la auditoría de atrición.


In [ ]:
raw['tipo_entrevista']=numeric_enasem(raw['tipent'])
directas=raw[raw.tipo_entrevista.isin([1,2])].copy()
mayores=filter_population(raw, min_age=60, interview_types=(1,2))
resumen=pd.DataFrame([
    {'criterio':'Entrevistas originales (incluye sustituto/muerte)','filas':len(raw),'personas':raw.id.nunique()},
    {'criterio':'Solo entrevistas directas','filas':len(directas),'personas':directas.id.nunique()},
    {'criterio':'Directas edad >=60 en esa ronda','filas':len(mayores),'personas':mayores.id.nunique()},
])
display(resumen)


## Comparación del panel de seis olas (sensibilidad, no baseline)
Si exiges >=60 en **cada** ronda, seleccionas personas de edad avanzada ya en 2001. Analizar la restricción antes de decidir.


In [ ]:
ids_seis=balanced_panel_ids(mayores,n_rounds=6)
panel_seis=mayores[mayores.id.isin(ids_seis)].copy()
print('Panel con seis rondas directas y edad>=60 en cada ronda:',panel_seis.id.nunique(),'personas')
print('Personas observadas por ronda (no balanceado):')
display(mayores.groupby(['ronda','ano']).agg(personas=('id','nunique'),filas=('id','size')).reset_index())
print(panel_quality(mayores))


## Próxima decisión CP1
Comparar con cohorte elegible a los 60 al inicio y con panel abierto de riesgo. Dejar por escrito población objetivo, sesgo de selección y uso de ponderadores.
